# Lecture 27 Worksheet: Recurrent Networks, Sequences and Memory
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`. Tasks 1 to 4 use numpy only. Tasks 5 and 6 use TensorFlow, which Colab already has.

## Task 1: One recurrent cell, over a whole sequence
The formula from slide 6: `h = tanh(W_xh * x + W_hh * h_old + b)`. Run it over the orders 1, 2, 3 from the lecture and compare with your paper answers (0.462, 0.829, 0.950).

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

W_xh, W_hh, b_h = 0.5, 0.4, 0.0     # the made-up weights from slide 8
W_hy, b_y = 2.0, 0.5

def rnn_forward(x, W_xh, W_hh, b_h, h0=0.0):
    h = h0
    memories = []
    for x_t in x:
        z = W_hh * h + W_xh * x_t + b_h     # old memory and today's input
        h = np.tanh(z)                        # squash
        memories.append(h)
    return np.array(memories)

h = rnn_forward([1, 2, 3], W_xh, W_hh, b_h)
y = W_hy * h[-1] + b_y
print("memories:", h, " output:", round(y, 2))
assert np.allclose(h, [0.462, 0.829, 0.950], atol=0.001) and abs(y - 2.40) < 0.01; print("OK")

memories: [0.462 0.829 0.95 ]  output: 2.4
OK


## Task 2: Does the order matter?
Feed the same three numbers in a different order and compare the last memory.

In [2]:
for seq in ([1, 2, 3], [3, 2, 1], [0, 0, 3], [3, 3, 3]):
    hs = rnn_forward(seq, W_xh, W_hh, b_h)
    print(seq, "-> memories", hs, " output", round(W_hy * hs[-1] + b_y, 2))

h_up = rnn_forward([1, 2, 3], W_xh, W_hh, b_h)[-1]
h_down = rnn_forward([3, 2, 1], W_xh, W_hh, b_h)[-1]
print("last memory, rising vs falling:", round(h_up, 3), round(h_down, 3))
assert abs(h_up - h_down) > 0.1; print("OK")      # choose a number that shows they differ clearly

[1, 2, 3] -> memories [0.462 0.829 0.95 ]  output 2.4
[3, 2, 1] -> memories [0.905 0.877 0.691]  output 1.88
[0, 0, 3] -> memories [0.    0.    0.905]  output 2.31
[3, 3, 3] -> memories [0.905 0.953 0.955]  output 2.41
last memory, rising vs falling: 0.95 0.691
OK


## Task 3: Count the parameters
A `SimpleRNN` with `u` units on inputs with `d` features has `u*d` input weights, `u*u` memory weights and `u` biases.

In [3]:
import tensorflow as tf
L = tf.keras.layers

def rnn_params(d, u):
    return u * d + u * u + u

print(rnn_params(1, 1), rnn_params(1, 8), rnn_params(1, 16), rnn_params(3, 16))
assert [rnn_params(1, 1), rnn_params(1, 8), rnn_params(3, 16)] == [3, 80, 320]

# confirm with Keras: a 14-day window, 1 feature, 16 units
m = tf.keras.Sequential([L.Input((14, 1)), L.SimpleRNN(16)])
print("Keras says:", m.count_params())
assert m.count_params() == rnn_params(1, 16)

# a dense layer on the flattened window has one weight per day per neuron
dense14 = 14 * 16 + 16
dense60 = 60 * 16 + 16
print("dense, 14-day window:", dense14, "  dense, 60-day window:", dense60, "  SimpleRNN(16), any window:", rnn_params(1, 16))
assert dense60 == 976; print("OK")

3 80 288 320
Keras says: 288
dense, 14-day window: 240   dense, 60-day window: 976   SimpleRNN(16), any window: 288
OK


## Task 4: Turn a series into windows
We generate 800 days of orders: a weekly pattern, a slow upward trend and some random noise. Then we cut it into windows of 14 days, each with the next day as the answer. Keras wants the shape `(n, 14, 1)`.

In [4]:
rs = np.random.RandomState(7)
t = np.arange(800)
week = np.array([80, 85, 90, 95, 120, 150, 140.])      # Mon ... Sun
series = week[t % 7] + 0.05 * t + rs.randn(800) * 6

W = 14
def make_windows(s, W):
    X = np.stack([s[i:i+W] for i in range(len(s) - W)])   # each row: W days
    y = s[W:]                                              # the day after each window
    return X, y

X, y = make_windows(series, W)
X3 = X[..., None]                       # add the 'features' axis
print(X.shape, X3.shape, y.shape)
assert X3.shape == (786, 14, 1) and y.shape == (786,); print("OK")

(786, 14) (786, 14, 1) (786,)
OK


## Task 5: Does the RNN beat simple methods?
Compare four predictions of tomorrow's orders on the last 186 windows: yesterday's value, the same day last week, linear regression on the window, and a SimpleRNN. Lower error is better.

In [5]:
from sklearn.linear_model import LinearRegression
X_tr, y_tr, X_te, y_te = X[:600], y[:600], X[600:], y[600:]
mae = lambda p: float(np.mean(np.abs(p - y_te)))

yesterday = X_te[:, -1]            # the last day in the window
same_day_last_week = X_te[:, -7]   # 7 days before tomorrow = 7th from the end
lr = LinearRegression().fit(X_tr, y_tr)
print("yesterday", round(mae(yesterday), 1), "| same day last week", round(mae(same_day_last_week), 1),
      "| linear regression", round(mae(lr.predict(X_te)), 1))

tf.keras.utils.set_random_seed(0)
mu, sd = X_tr.mean(), X_tr.std()
model = tf.keras.Sequential([L.Input((14, 1)), L.SimpleRNN(16), L.Dense(1)])   # 16 units
model.compile("adam", "mse")
model.fit(((X_tr - mu) / sd)[..., None], (y_tr - mu) / sd, epochs=40, batch_size=32, verbose=0)
p_rnn = model.predict(((X_te - mu) / sd)[..., None], verbose=0)[:, 0] * sd + mu
print("SimpleRNN", round(mae(p_rnn), 1))

assert mae(lr.predict(X_te)) < mae(yesterday) and mae(p_rnn) < mae(yesterday); print("OK")

yesterday 21.4 | same day last week 6.6 | linear regression 5.5
SimpleRNN 6.7
OK


**Question (write 2 sentences):** which method would you use for this series in a real project, and why?

## Task 6: How far back can the RNN remember?
Sequences of random numbers. The label says whether ONE chosen value is positive. In `first` the answer is the first number of the sequence; in `last` it is the last. Same network, same training.

In [6]:
def make_data(N, T, which, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    pos = 0 if which == "first" else -1          # index of the number that decides the label
    return x, (x[:, pos, 0] > 0).astype("float32")

def run(T, which, seed=0):
    tf.keras.utils.set_random_seed(seed)
    xtr, ytr = make_data(2000, T, which, seed)
    xte, yte = make_data(1000, T, which, seed + 100)
    m = tf.keras.Sequential([L.Input((T, 1)), L.SimpleRNN(16), L.Dense(1, activation="sigmoid")])
    m.compile(tf.keras.optimizers.Adam(0.01), "binary_crossentropy", metrics=["accuracy"])
    m.fit(xtr, ytr, epochs=15, batch_size=64, verbose=0)
    return m.evaluate(xte, yte, verbose=0)[1]

acc_first_10 = run(10, "first")
acc_first_40 = run(40, "first")
acc_last_40 = run(40, "last")
print("answer in FIRST value, 10 steps:", round(acc_first_10, 2))
print("answer in FIRST value, 40 steps:", round(acc_first_40, 2))
print("answer in LAST  value, 40 steps:", round(acc_last_40, 2))
assert acc_last_40 > 0.9 and acc_first_10 > 0.9; print("OK")

answer in FIRST value, 10 steps: 0.99
answer in FIRST value, 40 steps: 0.94
answer in LAST  value, 40 steps: 0.99
OK


**Question (write 2 sentences):** what do the three numbers tell you about how an RNN keeps early versus recent information?

## Task 7: Your own project
In 3 to 4 sentences: does your project data come in order (time or position)? If yes, which type of RNN from slide 14 fits (one-to-one, one-to-many, many-to-one, many-to-many)? What simple baseline would you try before any RNN?

*Your answer:*